In [52]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [53]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [54]:

# DATA_PATH = "EURUSD_5_MIN.csv"
eurusd = pd.read_csv("EURUSD_5_Min.csv")
eurjpy = pd.read_csv('EURJPY_5_Min.csv')
# data = add_all_ta_features(data,'open','close','low','high','volume')


In [55]:
# data_1=ta.add_trend_ta(eurusd,'high','low','close')
# data_2=ta.add_momentum_ta(eurusd,'high','low','close','volume')
# data_3=ta.add_volatility_ta(eurusd,'high','low','close','volume')
from ta.trend import macd,cci,adx,macd_signal
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
eur_rsi = rsi(eurusd['close'],14)
# rsi.dropna(axis=0,inplace=True)
eur_cci = cci(eurusd['high'],eurusd['low'],eurusd['close'],14)
# cci.dropna(axis=0,inplace=True)
eur_adx = adx(eurusd['high'],eurusd['low'],eurusd['close'])
# adx.dropna(axis=0,inplace=True)
eur_macd = macd(eurusd['close'])
# macd.dropna(axis=0,inplace=True)
eur_macd_signal = macd_signal(eurusd['close'])
# macd_signal.dropna(axis=0,inplace=True)
eur_stochrsi_d = stochrsi_d(eurusd['close'])
eur_stochrsi_k = stochrsi_k(eurusd['close'])
eur_stochrsi = stochrsi(eurusd['close'])
# stochrsi.dropna(axis=0,inplace=True)
pd1 = eurusd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
pd1['rsi'] = eur_rsi
pd1['cci'] = eur_cci
pd1['adx'] = eur_adx
pd1['macd'] = eur_macd
pd1['macd_signal'] = eur_macd_signal
pd1['stochrsi_k'] = eur_stochrsi_k
pd1['stochrsi_d'] = eur_stochrsi_d
pd1['stochrsi'] = eur_stochrsi
print(pd1.columns)

Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'macd', 'macd_signal', 'stochrsi_k', 'stochrsi_d', 'stochrsi'],
      dtype='object')


In [56]:
# data_1=ta.add_trend_ta(eurjpy,'high','low','close')
# data_2=ta.add_momentum_ta(eurjpy,'high','low','close','volume')
# data_3=ta.add_volatility_ta(eurjpy,'high','low','close','volume')
from ta.trend import macd,cci,adx,macd_signal
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
jpy_rsi = rsi(eurjpy['close'],14)
# rsi.dropna(axis=0,inplace=True)
jpy_cci = cci(eurjpy['high'],eurjpy['low'],eurjpy['close'],14)
# cci.dropna(axis=0,inplace=True)
jpy_adx = adx(eurjpy['high'],eurjpy['low'],eurjpy['close'])
# adx.dropna(axis=0,inplace=True)
jpy_macd = macd(eurjpy['close'])
# macd.dropna(axis=0,inplace=True)
jpy_macd_signal = macd_signal(eurjpy['close'])
# macd_signal.dropna(axis=0,inplace=True)
jpy_stochrsi_d = stochrsi_d(eurjpy['close'])
jpy_stochrsi_k = stochrsi_k(eurjpy['close'])
jpy_stochrsi = stochrsi(eurjpy['close'])
# stochrsi.dropna(axis=0,inplace=True)
pd2 = eurjpy.iloc[:,1:7].copy(deep=True) # iloc[row,column]
pd2['rsi'] = jpy_rsi
pd2['cci'] = jpy_cci
pd2['adx'] = jpy_adx
pd2['macd'] = jpy_macd
pd2['macd_signal'] = jpy_macd_signal
pd2['stochrsi_d'] = jpy_stochrsi_d
pd2['stochrsi_k'] = jpy_stochrsi_k
pd2['stochrsi'] = jpy_stochrsi
print(pd1.columns)

Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'macd', 'macd_signal', 'stochrsi_k', 'stochrsi_d', 'stochrsi'],
      dtype='object')


In [57]:
data = pd.concat([pd1,pd2])
data.dropna(axis=0,inplace=True)
print(data.columns)
print(data.count())

Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'macd', 'macd_signal', 'stochrsi_k', 'stochrsi_d', 'stochrsi'],
      dtype='object')
symbol         10079
open           10079
high           10079
low            10079
close          10079
volume         10079
rsi            10079
cci            10079
adx            10079
macd           10079
macd_signal    10079
stochrsi_k     10079
stochrsi_d     10079
stochrsi       10079
dtype: int64


In [58]:
data['RSI_1'] = np.where(data['rsi'] < 40, 1, np.where(data['rsi'] > 60, 2, 0))
# data['MACD_1'] = np.where(data['macd'] < data['macd_signal'], 2, np.where(data['macd'] > data['macd_signal'], 1, 0))
# data['CCI_1'] = np.where(data['cci'] < -80, 1, np.where(data['cci'] > 80, 2, 0))
data['ADX_1'] = np.where(data['adx'] > 25, 1, 0)
# conditions_3 = (data['stochrsi'] > 0.75) & (data['stochrsi_k'] < data['stochrsi_d'])
# conditions_4 = (data['stochrsi'] < 0.25) & (data['stochrsi_k'] > data['stochrsi_d'])
# data['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
conditions_1 = (data['ADX_1'] == 1)  #(data['ADX_1'] == 1) & (data['MACD_1'] == 1) #(data['STOCH.RSI'] == 1) # & (data['CCI_1'] == 1) & (data['MACD_1'] == 1) (data['ADX_1'] == 1) & (data['RSI_1'] == 1) & 
conditions_2 = (data['ADX_1'] == 1) #(data['ADX_1'] == 1) & (data['MACD_1'] == 2) #(data['STOCH.RSI'] == 2) # & (data['CCI_1'] == 2) & (data['MACD_1'] == 2) & (data['ADX_1'] == 1)  (data['RSI_1'] == 2) & 
data['Prediction'] = np.where(conditions_1 & (data['open'] < data['close']), 1,
                              np.where(conditions_2 & (data['open'] > data['close']), 2, 0))


In [59]:
# data.drop(axis=1,labels=['rsi','cci','adx','macd','macd_signal','stochrsi','stochrsi_k','stochrsi_d'],inplace=True)
data.drop(axis=1,labels=['RSI_1'],inplace=True)

In [60]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.head())
print(data.shape)


          symbol     open     high      low    close  volume        rsi  \
0  FX_IDC:EURUSD  1.08478  1.08480  1.08463  1.08470  2837.0  72.618496   
1  FX_IDC:EURUSD  1.08471  1.08476  1.08458  1.08471  3863.0  72.919624   
2  FX_IDC:EURUSD  1.08472  1.08480  1.08444  1.08449  3301.0  57.847187   
3  FX_IDC:EURUSD  1.08456  1.08458  1.08434  1.08439  3699.0  52.531928   
4  FX_IDC:EURUSD  1.08439  1.08443  1.08418  1.08425  3487.0  46.139992   

          cci        adx      macd  macd_signal  stochrsi_k  stochrsi_d  \
0  150.645342  41.265595  0.000156     0.000112    0.935349    0.935348   
1  113.218606  41.923274  0.000165     0.000123    0.874758    0.932423   
2   63.500440  40.854860  0.000152     0.000128    0.610991    0.807033   
3    4.601077  38.889838  0.000132     0.000129    0.342308    0.609352   
4  -66.246057  36.421900  0.000104     0.000124    0.069566    0.340955   

   stochrsi  ADX_1  Prediction  
0  0.806048      1           2  
1  0.818225      1           0  

In [61]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


[0 1 2]


In [62]:
print(data['Prediction'].value_counts())
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.head())
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.2, random_state = 24)



Prediction
0    6136
1    1996
2    1947
Name: count, dtype: int64
Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'macd', 'macd_signal', 'stochrsi_k', 'stochrsi_d', 'stochrsi', 'ADX_1',
       'Prediction'],
      dtype='object')
         rsi         cci        adx      macd  macd_signal  stochrsi_k  \
0  72.618496  150.645342  41.265595  0.000156     0.000112    0.935349   
1  72.919624  113.218606  41.923274  0.000165     0.000123    0.874758   
2  57.847187   63.500440  40.854860  0.000152     0.000128    0.610991   
3  52.531928    4.601077  38.889838  0.000132     0.000129    0.342308   
4  46.139992  -66.246057  36.421900  0.000104     0.000124    0.069566   

   stochrsi_d  stochrsi  ADX_1  
0    0.935348  0.806048      1  
1    0.932423  0.818225      1  
2    0.807033  0.208699      1  
3    0.609352  0.000000      1  
4    0.340955  0.000000      1  
rsi            10079
cci            10079
adx            10079
macd           10079
mac

In [73]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=6,min_child_weight = 3)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by XGBoost Classifier: 99.76435569887138
Accuracy on test data by XGBoost Classifier: 92.26190476190477


In [74]:

svm_model = SVC()
svm_model.fit(X_train, y_train)
preds = svm_model.predict(X_test)
 
print(f"Accuracy on train data by SVM Classifier\
: {accuracy_score(y_train, svm_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by SVM Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by SVM Classifier: 80.007441398983
Accuracy on test data by SVM Classifier: 80.65476190476191


In [75]:
# Training and testing Random Forest Classifier
rf_model = RandomForestClassifier(random_state=18)
rf_model.fit(X_train, y_train)
preds = rf_model.predict(X_test)
print(f"Accuracy on train data by Random Forest Classifier\
: {accuracy_score(y_train, rf_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by Random Forest Classifier\
: {accuracy_score(y_test, preds)*100}")
 


Accuracy on train data by Random Forest Classifier: 100.0
Accuracy on test data by Random Forest Classifier: 92.11309523809523


In [66]:
kn_model = KNeighborsClassifier(n_neighbors=3)
kn_model.fit(X_train, y_train)
preds = kn_model.predict(X_test)
print(f"Accuracy on train data by K Neighbors Classifier\
: {accuracy_score(y_train, kn_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by K Neighbors Classifier\
: {accuracy_score(y_test, preds)*100}")

Accuracy on train data by K Neighbors Classifier: 89.39600644921245
Accuracy on test data by K Neighbors Classifier: 79.7123015873016


In [67]:
# # Training the models on whole data
# final_svm_model = SVC()
# final_rf_model = RandomForestClassifier(random_state=18)
# final_kn_model = KNeighborsClassifier(n_neighbors=3)
final_xgb_model = XGBClassifier()
# final_svm_model.fit(X, y)
# final_rf_model.fit(X, y)
# final_kn_model.fit(X, y)
final_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [82]:
# from ..TradingDataGenerate.main import TvDatafeed
# from ....TradingDataGenerate.main import TvDatafeed
from TradingDataGeneration.TradingDataGenerate import main
s = main.TvDatafeed('mageshragav1@gmail.com','Magesh1@')
m = s.get_hist(symbol='EURJPY',exchange='FX_IDC',interval=main.Interval.in_5_minute,n_bars=9999,extended_session=True)
# m.to_csv('EURGBP_1_Min.csv')
data_1 = {'rsi':[29.68], 'cci':[-249], 'adx':[18], 'macd':[-0.00007], 'macd_signal':[0.00003], 'stochrsi_k': [0.00], 'stochrsi_d':[16.64], 'stochrsi':[0.023], 'ADX_1':[0]}
final_xgb_model.predict(pd.DataFrame(data_1))

ModuleNotFoundError: No module named 'TradingDataGeneration'